In [17]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
%matplotlib inline

# Load data
df = pd.read_csv('../data/raw/training.csv')
print("Shape:", df.shape)
df.head()

Shape: (95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [ ]:
# Cell 2 — Structure
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

print("\n=== COLUMN TYPES ===")
print(df.dtypes)

print("\n=== FIRST 5 ROWS ===")
df.head()

In [ ]:
# Cell 3 — Basic info
df.info()

In [ ]:
# Cell 4 — Summary statistics
df.describe()

In [ ]:
# Cell 5 — Deeper look at key financial columns
print("Amount stats:")
print(df['Amount'].describe())
print(f"\nNegative amounts (credits): {(df['Amount'] < 0).sum():,}")
print(f"Positive amounts (debits): {(df['Amount'] > 0).sum():,}")

print("\nValue stats:")
print(df['Value'].describe())

print("\nFraud rate:")
print(df['FraudResult'].value_counts())
print(f"Fraud %: {df['FraudResult'].mean()*100:.2f}%")

In [ ]:
# Cell 6 — Numerical distributions
numerical_cols = ['Amount', 'Value']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(numerical_cols):
    # Histogram
    axes[i, 0].hist(df[col], bins=50, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f'Distribution of {col}')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frequency')
    
    # Log-scale histogram to handle skewness
    axes[i, 1].hist(df[col][df[col] > 0], bins=50, 
                     color='coral', edgecolor='white', log=True)
    axes[i, 1].set_title(f'Log-scale Distribution of {col} (positive only)')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.savefig('../data/processed/numerical_distributions.png', dpi=150)
plt.show()

In [ ]:
# Cell 7 — Parse datetime and extract time features
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])
df['TransactionHour'] = df['TransactionStartTime'].dt.hour
df['TransactionDayOfWeek'] = df['TransactionStartTime'].dt.dayofweek
df['TransactionMonth'] = df['TransactionStartTime'].dt.month

# Distribution of transactions by hour
plt.figure(figsize=(12, 4))
df['TransactionHour'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Transaction Volume by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Categorical distributions
categorical_cols = ['ProductCategory', 'ChannelId', 'ProviderId', 'PricingStrategy']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[i].bar(value_counts.index.astype(str), value_counts.values, color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/categorical_distributions.png', dpi=150)
plt.show()

In [ ]:
# Cell 9 — Fraud rate by category (very important!)
fraud_by_category = df.groupby('ProductCategory')['FraudResult'].agg(['mean', 'sum', 'count'])
fraud_by_category.columns = ['FraudRate', 'FraudCount', 'TotalTransactions']
fraud_by_category = fraud_by_category.sort_values('FraudRate', ascending=False)

print("Fraud rate by Product Category:")
print(fraud_by_category)

fraud_by_category['FraudRate'].plot(kind='bar', color='coral', figsize=(10, 4))
plt.title('Fraud Rate by Product Category')
plt.ylabel('Fraud Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Correlation heatmap
corr_cols = ['Amount', 'Value', 'TransactionHour', 
             'TransactionDayOfWeek', 'TransactionMonth', 'FraudResult']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, square=True)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Cell 11 — Correlation with FraudResult specifically
print("Correlation with FraudResult:")
print(df[corr_cols].corr()['FraudResult'].sort_values(ascending=False))

In [ ]:
# Cell 12 — Missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("Missing Values Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize
if missing_df['Missing Count'].sum() > 0:
    missing_df[missing_df['Missing Count'] > 0]['Missing %'].plot(
        kind='bar', color='coral', figsize=(10, 4))
    plt.title('Missing Values by Column (%)')
    plt.ylabel('Missing %')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in this dataset.")

In [ ]:
# Cell 13 — Outlier detection via boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='Amount', ax=axes[0])
axes[0].set_title('Boxplot: Transaction Amount')
axes[0].set_ylabel('Amount')

df[df['Value'] > 0].boxplot(column='Value', ax=axes[1])
axes[1].set_title('Boxplot: Transaction Value (positive only)')
axes[1].set_ylabel('Value')

plt.tight_layout()
plt.savefig('../data/processed/outlier_boxplots.png', dpi=150)
plt.show()

# Quantify outliers using IQR
for col in ['Amount', 'Value']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers):,} outliers ({len(outliers)/len(df)*100:.1f}%)")

In [ ]:
# Cell 14 — Customer-level aggregation preview
snapshot_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)

customer_stats = df.groupby('CustomerId').agg(
    NumTransactions=('TransactionId', 'count'),
    TotalAmount=('Amount', 'sum'),
    AvgAmount=('Amount', 'mean'),
    LastTransaction=('TransactionStartTime', 'max')
).reset_index()

customer_stats['Recency'] = (snapshot_date - customer_stats['LastTransaction']).dt.days

print(f"Total unique customers: {customer_stats.shape[0]:,}")
print("\nCustomer-level statistics:")
print(customer_stats[['NumTransactions', 'TotalAmount', 'AvgAmount', 'Recency']].describe())

## EDA Summary — Top 5 Insights

### Insight 1: Severe Class Imbalance in Fraud Labels
The FraudResult column shows fewer than 1% of transactions are fraudulent.
This means accuracy is a misleading metric — a model that always predicts
"not fraud" would score ~99% accuracy. We must use AUC-ROC, Precision-Recall,
and F1 Score to evaluate models. We may need oversampling (SMOTE) or
class-weight adjustments during training.

### Insight 2: Transaction Amount Is Highly Right-Skewed with Outliers
Most transactions are small amounts, but there are extreme high-value
outliers (some exceeding 10x the 75th percentile). Log transformation or
robust scaling will be necessary during feature engineering to prevent these
from dominating model training. Notably, Amount includes negative values
(credits/refunds), which is meaningful behavioral information.

### Insight 3: Transaction Timing Shows Behavioral Patterns
Transactions cluster at specific hours of the day, suggesting customer
activity patterns. These time-based features (hour, day of week) may serve
as behavioral signals in our credit risk model — for example, late-night
transactions may correlate with different risk profiles.

### Insight 4: Product Category Drives Fraud Rate Significantly
Fraud rates vary substantially across ProductCategory values. Certain
categories have disproportionately high fraud rates, making ProductCategory
an important feature for both fraud detection and credit risk prediction.
This also motivates one-hot encoding of this feature carefully.

### Insight 5: Customer Behavior Is Highly Variable — RFM Signals Will Matter
At the customer level, transaction frequency ranges from 1 to hundreds of
transactions. Recency varies widely. This high variability confirms that
RFM-based segmentation (Task 4) will produce meaningfully distinct customer
clusters that can serve as a credible proxy for credit risk.

### Implications for Modeling
- Target variable: We cannot use FraudResult directly as a credit risk label.
  We will engineer a proxy via RFM clustering (Task 4).
- Feature scaling is essential before clustering and for linear models.
- Categorical encoding is needed for ProductCategory, ChannelId, ProviderId.
- Outlier handling strategy: cap or log-transform Amount and Value.
- Time features (hour, day, month) should be extracted and included.